# Practical P19: Building a Reusable API Client Class
**Learning Outcome**: Build a reusable, production-ready AI API client class.

## Part 1: Design of the `AIClient` class
Object-Oriented Programming (OOP) allows us to encapsulate API endpoints, authorization tokens, costs, and retry rules inside a single object instance.


In [ ]:
import requests
import json
import time
import logging

class AIClient:
    def __init__(self, api_key, model='gpt-4o-mini', max_tokens=100, cost_per_1k_tokens=0.002):
        self.api_key = api_key
        self.model = model
        self.max_tokens = max_tokens
        self.cost_per_1k_tokens = cost_per_1k_tokens
        self.total_tokens_used = 0
        self.total_cost_usd = 0.0
        
    def _build_headers(self):
        return {
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {self.api_key}'
        }
        
    def _calculate_cost(self, tokens):
        self.total_tokens_used += tokens
        self.total_cost_usd += (tokens / 1000.0) * self.cost_per_1k_tokens
        
    def call(self, prompt):
        # Simulating a local API mock for clean classroom testing
        time.sleep(0.1) # latency delay simulation
        mock_output = f"[Response from {self.model}] Answer to '{prompt}'"
        tokens = len(prompt.split()) + len(mock_output.split())
        
        # Accumulate metrics
        self._calculate_cost(tokens)
        
        return {
            'model': self.model,
            'response': mock_output,
            'tokens': tokens
        }


## Part 2: Testing the Client Instance
Let's create an instance of our `AIClient` and make several calls.


In [ ]:
client = AIClient(api_key='sk-test-key-9999', model='gpt-4o')

r1 = client.call('Tell me a programming joke.')
r2 = client.call('What is Python?')

print('Response 1:', r1['response'])
print('Response 2:', r2['response'])
print(f'Total Tokens Used: {client.total_tokens_used}')
print(f'Total Cost accrued: ${client.total_cost_usd:.6f}')


## Hands-On Exercise
**Task**: Extend the `AIClient` class to track prompt and completion tokens separately. Define distinct cost ratios:
- Input Cost: $0.0015 / 1k tokens
- Output Cost: $0.0020 / 1k tokens
Write a script that processes a bulk list of 5 prompts and outputs a final Cost Statement Report.


In [ ]:
# TODO: Implement updated AIClient with input/output split costs
class AdvancedAIClient(AIClient):
    def __init__(self, api_key, model='gpt-4o-mini'):
        super().__init__(api_key, model)
        self.input_cost_1k = 0.0015
        self.output_cost_1k = 0.0020
        
    def call(self, prompt):
        # Count prompt words as input tokens
        input_tokens = len(prompt.split())
        # Mock answer
        answer = f"Processed: {prompt[:20]}..."
        output_tokens = len(answer.split())
        
        # Cost calculation
        cost = (input_tokens / 1000.0) * self.input_cost_1k + (output_tokens / 1000.0) * self.output_cost_1k
        self.total_tokens_used += (input_tokens + output_tokens)
        self.total_cost_usd += cost
        
        return {'text': answer, 'cost': cost}

adv_client = AdvancedAIClient('sk-mock-key')
prompts = [
    'Prompt Alpha', 'Prompt Beta', 'Prompt Gamma', 'Prompt Delta', 'Prompt Epsilon'
]

for p in prompts:
    res = adv_client.call(p)
    print(f"Prompt: '{p}' -> Cost: ${res['cost']:.6f}")
    
print(f'Accumulated session cost: ${adv_client.total_cost_usd:.6f}')
